In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import os

# Define the file paths as commonly found in the Kaggle environment
base_path = "/kaggle/input/house-prices-advanced-regression-techniques/"
train_file_path = os.path.join(base_path, "train.csv")
test_file_path = os.path.join(base_path, "test.csv")

# 1. Load the training data into train_df
try:
    print(f"Attempting to load training data from: {train_file_path}")
    train_df = pd.read_csv(train_file_path)
    print("Training data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Training file not found at {train_file_path}. Please ensure the path is correct.")
    # Create an empty DataFrame to prevent errors later if the file is missing
    train_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading training data: {e}")
    train_df = pd.DataFrame()

# 2. Load the test data into test_df
try:
    print(f"Attempting to load test data from: {test_file_path}")
    test_df = pd.read_csv(test_file_path)
    print("Test data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Test file not found at {test_file_path}. Please ensure the path is correct.")
    test_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading test data: {e}")
    test_df = pd.DataFrame()


# 3. Display the first 5 rows of train_df if it was loaded
if not train_df.empty:
    print("\n--- First 5 rows of train_df ---")
    print(train_df.head())
    print("\nShape of train_df:", train_df.shape)
    print("Shape of test_df:", test_df.shape)
else:
    print("\nCannot display train_df.head() because the DataFrame is empty due to a loading error.")


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'Inter'

# Define the file paths as commonly found in the Kaggle environment
base_path = "/kaggle/input/house-prices-advanced-regression-techniques/"
train_file_path = os.path.join(base_path, "train.csv")
test_file_path = os.path.join(base_path, "test.csv")

# 1. Load the training data into train_df
try:
    print(f"Attempting to load training data from: {train_file_path}")
    train_df = pd.read_csv(train_file_path)
    print("Training data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Training file not found at {train_file_path}. Please ensure the path is correct.")
    # Create an empty DataFrame to prevent errors later if the file is missing
    train_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading training data: {e}")
    train_df = pd.DataFrame()

# 2. Load the test data into test_df
try:
    print(f"Attempting to load test data from: {test_file_path}")
    test_df = pd.read_csv(test_file_path)
    print("Test data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Test file not found at {test_file_path}. Please ensure the path is correct.")
    test_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading test data: {e}")
    test_df = pd.DataFrame()


# 3. Perform Exploratory Data Analysis (EDA) on train_df
if not train_df.empty:
    print("\n" + "="*70)
    print("Starting Comprehensive Exploratory Data Analysis (EDA) on train_df")
    print("="*70)

    # Basic Structure and Statistics
    print("\n--- 3a. DataFrame Structure and Head ---")
    print("First 5 rows of train_df:")
    print(train_df.head())
    print("\nShape of train_df:", train_df.shape)
    print("Shape of test_df:", test_df.shape)

    print("\n--- 3b. train_df.info() (Non-null values and Data Types) ---")
    train_df.info()

    print("\n--- 3c. train_df.describe() (Descriptive Statistics for Numeric Columns) ---")
    # Display statistics for the target variable (SalePrice)
    print(train_df['SalePrice'].describe())

    # 1) Histogram of SalePrice (Raw and Log-Transformed)
    print("\n--- 3d. SalePrice Distribution (Visualizations) ---")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Distribution of House SalePrice', fontsize=18)

    # Raw SalePrice Histogram
    sns.histplot(train_df['SalePrice'], bins=50, kde=True, ax=ax1, color='teal')
    ax1.set_title('Raw SalePrice Distribution (Right-Skewed)')
    ax1.set_xlabel('SalePrice ($)', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.ticklabel_format(style='plain', axis='x')

    # Log-Transformed SalePrice Histogram (using log(1+x) to handle potential zeros, though SalePrice > 0)
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])
    sns.histplot(train_df['SalePrice_log'], bins=50, kde=True, ax=ax2, color='darkorange')
    ax2.set_title('Log-Transformed SalePrice Distribution (Closer to Normal)')
    ax2.set_xlabel('Log(1 + SalePrice)', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    # Remove the temporary log column
    train_df = train_df.drop(columns=['SalePrice_log'])


    # 2) Heatmap of Correlations among Top 10 Numeric Features related to SalePrice
    print("\n--- 3e. Correlation Heatmap (Top 10 Features vs SalePrice) ---")
    # Calculate correlations with SalePrice for all numeric columns
    corrmat = train_df.corr(numeric_only=True)
    
    # Sort features by absolute correlation value with 'SalePrice'
    # We take the top 10 most correlated features (excluding 'SalePrice' itself) and add 'SalePrice' back.
    top_features_corr = corrmat['SalePrice'].abs().sort_values(ascending=False)
    # Select the top 10 features (indices 1 to 10 are the top 10 non-SalePrice features)
    top_10_features = top_features_corr[1:11].index.tolist() + ['SalePrice']
    
    # Calculate correlation matrix for the top features
    corr_matrix_top = train_df[top_10_features].corr(numeric_only=True)

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix_top, annot=True, fmt=".2f", cmap='coolwarm', 
                linewidths=0.5, linecolor='black', cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Correlation Heatmap: Top 10 Features vs. SalePrice', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


    # 3) Boxplots for categorical variables (e.g., Neighborhood vs SalePrice)
    print("\n--- 3f. Boxplot: Neighborhood vs SalePrice ---")
    plt.figure(figsize=(18, 7))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df, palette='Spectral')
    plt.title('SalePrice Distribution Across Neighborhoods', fontsize=16)
    plt.xlabel('Neighborhood', fontsize=12)
    plt.ylabel('SalePrice ($)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()

else:
    print("\nSkipping EDA because train_df is empty due to a loading error.")


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'Inter'

# Define the file paths as commonly found in the Kaggle environment
base_path = "/kaggle/input/house-prices-advanced-regression-techniques/"
train_file_path = os.path.join(base_path, "train.csv")
test_file_path = os.path.join(base_path, "test.csv")

# 1. Load the training data into train_df
try:
    print(f"Attempting to load training data from: {train_file_path}")
    train_df = pd.read_csv(train_file_path)
    print("Training data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Training file not found at {train_file_path}. Please ensure the path is correct.")
    # Create an empty DataFrame to prevent errors later if the file is missing
    train_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading training data: {e}")
    train_df = pd.DataFrame()

# 2. Load the test data into test_df
try:
    print(f"Attempting to load test data from: {test_file_path}")
    test_df = pd.read_csv(test_file_path)
    print("Test data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Test file not found at {test_file_path}. Please ensure the path is correct.")
    test_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading test data: {e}")
    test_df = pd.DataFrame()


# 3. Perform Exploratory Data Analysis (EDA) on train_df
if not train_df.empty:
    print("\n" + "="*70)
    print("Starting Comprehensive Exploratory Data Analysis (EDA) on train_df")
    print("="*70)

    # Basic Structure and Statistics
    print("\n--- 3a. DataFrame Structure and Head ---")
    print("First 5 rows of train_df:")
    print(train_df.head())
    print("\nShape of train_df:", train_df.shape)
    print("Shape of test_df:", test_df.shape)

    print("\n--- 3b. train_df.info() (Non-null values and Data Types) ---")
    train_df.info()

    print("\n--- 3c. train_df.describe() (Descriptive Statistics for Numeric Columns) ---")
    # Display statistics for the target variable (SalePrice)
    print(train_df['SalePrice'].describe())

    # 1) Histogram of SalePrice (Raw and Log-Transformed)
    print("\n--- 3d. SalePrice Distribution (Visualizations) ---")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Distribution of House SalePrice', fontsize=18)

    # Raw SalePrice Histogram
    sns.histplot(train_df['SalePrice'], bins=50, kde=True, ax=ax1, color='teal')
    ax1.set_title('Raw SalePrice Distribution (Right-Skewed)')
    ax1.set_xlabel('SalePrice ($)', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.ticklabel_format(style='plain', axis='x')

    # Log-Transformed SalePrice Histogram (using log(1+x) to handle potential zeros, though SalePrice > 0)
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])
    sns.histplot(train_df['SalePrice_log'], bins=50, kde=True, ax=ax2, color='darkorange')
    ax2.set_title('Log-Transformed SalePrice Distribution (Closer to Normal)')
    ax2.set_xlabel('Log(1 + SalePrice)', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    # Remove the temporary log column
    train_df = train_df.drop(columns=['SalePrice_log'])


    # 2) Heatmap of Correlations among Top 10 Numeric Features related to SalePrice
    print("\n--- 3e. Correlation Heatmap (Top 10 Features vs SalePrice) ---")
    # Calculate correlations with SalePrice for all numeric columns
    corrmat = train_df.corr(numeric_only=True)
    
    # Sort features by absolute correlation value with 'SalePrice'
    # We take the top 10 most correlated features (excluding 'SalePrice' itself) and add 'SalePrice' back.
    top_features_corr = corrmat['SalePrice'].abs().sort_values(ascending=False)
    # Select the top 10 features (indices 1 to 10 are the top 10 non-SalePrice features)
    top_10_features = top_features_corr[1:11].index.tolist() + ['SalePrice']
    
    # Calculate correlation matrix for the top features
    corr_matrix_top = train_df[top_10_features].corr(numeric_only=True)

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix_top, annot=True, fmt=".2f", cmap='coolwarm', 
                linewidths=0.5, linecolor='black', cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Correlation Heatmap: Top 10 Features vs. SalePrice', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


    # 3) Boxplots for categorical variables (e.g., Neighborhood vs SalePrice)
    print("\n--- 3f. Boxplot: Neighborhood vs SalePrice ---")
    plt.figure(figsize=(18, 7))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df, palette='Spectral')
    plt.title('SalePrice Distribution Across Neighborhoods', fontsize=16)
    plt.xlabel('Neighborhood', fontsize=12)
    plt.ylabel('SalePrice ($)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()

    # 4. Data Splitting
    print("\n" + "="*70)
    print("4. Splitting Data into Training and Test Sets (80/20)")
    print("="*70)

    # Separate features (X) and target (y)
    X = train_df.drop('SalePrice', axis=1)
    y = train_df['SalePrice']

    # Perform 80/20 split (test_size=0.2 means 20% for testing)
    # We use random_state=42 for reproducibility
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Original train_df shape: {train_df.shape}")
    print(f"Features (X) shape: {X.shape}, Target (y) shape: {y.shape}")
    print("\n--- Split Shapes ---")
    print(f"X_train shape (80% features): {X_train.shape}")
    print(f"X_test shape (20% features): {X_test.shape}")
    print(f"y_train shape (80% target): {y_train.shape}")
    print(f"y_test shape (20% target): {y_test.shape}")
    
else:
    print("\nSkipping EDA and Splitting because train_df is empty due to a loading error.")


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer # Added for imputation

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'Inter'

# Define the file paths as commonly found in the Kaggle environment
base_path = "/kaggle/input/house-prices-advanced-regression-techniques/"
train_file_path = os.path.join(base_path, "train.csv")
test_file_path = os.path.join(base_path, "test.csv")

# 1. Load the training data into train_df
try:
    print(f"Attempting to load training data from: {train_file_path}")
    train_df = pd.read_csv(train_file_path)
    print("Training data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Training file not found at {train_file_path}. Please ensure the path is correct.")
    # Create an empty DataFrame to prevent errors later if the file is missing
    train_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading training data: {e}")
    train_df = pd.DataFrame()

# 2. Load the test data into test_df
try:
    print(f"Attempting to load test data from: {test_file_path}")
    test_df = pd.read_csv(test_file_path)
    print("Test data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Test file not found at {test_file_path}. Please ensure the path is correct.")
    test_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading test data: {e}")
    test_df = pd.DataFrame()


# 3. Perform Exploratory Data Analysis (EDA) on train_df
if not train_df.empty:
    print("\n" + "="*70)
    print("Starting Comprehensive Exploratory Data Analysis (EDA) on train_df")
    print("="*70)

    # Basic Structure and Statistics
    print("\n--- 3a. DataFrame Structure and Head ---")
    print("First 5 rows of train_df:")
    print(train_df.head())
    print("\nShape of train_df:", train_df.shape)
    print("Shape of test_df:", test_df.shape)

    print("\n--- 3b. train_df.info() (Non-null values and Data Types) ---")
    train_df.info()

    print("\n--- 3c. train_df.describe() (Descriptive Statistics for Numeric Columns) ---")
    # Display statistics for the target variable (SalePrice)
    print(train_df['SalePrice'].describe())

    # 1) Histogram of SalePrice (Raw and Log-Transformed)
    print("\n--- 3d. SalePrice Distribution (Visualizations) ---")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Distribution of House SalePrice', fontsize=18)

    # Raw SalePrice Histogram
    sns.histplot(train_df['SalePrice'], bins=50, kde=True, ax=ax1, color='teal')
    ax1.set_title('Raw SalePrice Distribution (Right-Skewed)')
    ax1.set_xlabel('SalePrice ($)', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.ticklabel_format(style='plain', axis='x')

    # Log-Transformed SalePrice Histogram (using log(1+x) to handle potential zeros, though SalePrice > 0)
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])
    sns.histplot(train_df['SalePrice_log'], bins=50, kde=True, ax=ax2, color='darkorange')
    ax2.set_title('Log-Transformed SalePrice Distribution (Closer to Normal)')
    ax2.set_xlabel('Log(1 + SalePrice)', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    # Remove the temporary log column
    train_df = train_df.drop(columns=['SalePrice_log'])


    # 2) Heatmap of Correlations among Top 10 Numeric Features related to SalePrice
    print("\n--- 3e. Correlation Heatmap (Top 10 Features vs SalePrice) ---")
    # Calculate correlations with SalePrice for all numeric columns
    corrmat = train_df.corr(numeric_only=True)
    
    # Sort features by absolute correlation value with 'SalePrice'
    # We take the top 10 most correlated features (excluding 'SalePrice' itself) and add 'SalePrice' back.
    top_features_corr = corrmat['SalePrice'].abs().sort_values(ascending=False)
    # Select the top 10 features (indices 1 to 10 are the top 10 non-SalePrice features)
    top_10_features = top_features_corr[1:11].index.tolist() + ['SalePrice']
    
    # Calculate correlation matrix for the top features
    corr_matrix_top = train_df[top_10_features].corr(numeric_only=True)

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix_top, annot=True, fmt=".2f", cmap='coolwarm', 
                linewidths=0.5, linecolor='black', cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Correlation Heatmap: Top 10 Features vs. SalePrice', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


    # 3) Boxplots for categorical variables (e.g., Neighborhood vs SalePrice)
    print("\n--- 3f. Boxplot: Neighborhood vs SalePrice ---")
    plt.figure(figsize=(18, 7))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df, palette='Spectral')
    plt.title('SalePrice Distribution Across Neighborhoods', fontsize=16)
    plt.xlabel('Neighborhood', fontsize=12)
    plt.ylabel('SalePrice ($)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()

    # 4. Data Splitting
    print("\n" + "="*70)
    print("4. Splitting Data into Training and Test Sets (80/20)")
    print("="*70)

    # Separate features (X) and target (y)
    X = train_df.drop('SalePrice', axis=1)
    y = train_df['SalePrice']

    # Perform 80/20 split (test_size=0.2 means 20% for testing)
    # We use random_state=42 for reproducibility
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Original train_df shape: {train_df.shape}")
    print(f"Features (X) shape: {X.shape}, Target (y) shape: {y.shape}")
    print("\n--- Split Shapes ---")
    print(f"X_train shape (80% features): {X_train.shape}")
    print(f"X_test shape (20% features): {X_test.shape}")
    print(f"y_train shape (80% target): {y_train.shape}")
    print(f"y_test shape (20% target): {y_test.shape}")

    # 5. Simple Linear Regression Model Training (Numeric Features Only)
    print("\n" + "="*70)
    print("5. Training a Linear Regression Model (Pre-processing Numeric Features)")
    print("="*70)
    
    # Select only numeric features for a simple initial model
    numeric_features = X_train.select_dtypes(include=np.number).columns
    
    X_train_numeric = X_train[numeric_features]
    X_test_numeric = X_test[numeric_features]

    print(f"Selected {len(numeric_features)} numeric features for modeling.")
    
    # Handle Missing Values: Impute NaNs with the median of each column
    imputer = SimpleImputer(strategy='median')
    
    # Fit imputer on X_train_numeric and transform both train and test sets
    X_train_imputed = imputer.fit_transform(X_train_numeric)
    X_test_imputed = imputer.transform(X_test_numeric)

    # Initialize and train the Linear Regression Model
    model = LinearRegression()
    print("Training Linear Regression model...")
    model.fit(X_train_imputed, y_train)
    print("Training complete.")

    # Evaluate the model on the training set (R-squared score)
    train_score = model.score(X_train_imputed, y_train)
    print(f"\nLinear Regression Training R^2 Score (Numeric Features Only): {train_score:.4f}")
    
else:
    print("\nSkipping EDA, Splitting, and Modeling because train_df is empty due to a loading error.")


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error # Added for evaluation metrics

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'Inter'

# Define the file paths as commonly found in the Kaggle environment
base_path = "/kaggle/input/house-prices-advanced-regression-techniques/"
train_file_path = os.path.join(base_path, "train.csv")
test_file_path = os.path.join(base_path, "test.csv")

# 1. Load the training data into train_df
try:
    print(f"Attempting to load training data from: {train_file_path}")
    train_df = pd.read_csv(train_file_path)
    print("Training data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Training file not found at {train_file_path}. Please ensure the path is correct.")
    # Create an empty DataFrame to prevent errors later if the file is missing
    train_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading training data: {e}")
    train_df = pd.DataFrame()

# 2. Load the test data into test_df
try:
    print(f"Attempting to load test data from: {test_file_path}")
    test_df = pd.read_csv(test_file_path)
    print("Test data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Test file not found at {test_file_path}. Please ensure the path is correct.")
    test_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading test data: {e}")
    test_df = pd.DataFrame()


# 3. Perform Exploratory Data Analysis (EDA) on train_df
if not train_df.empty:
    print("\n" + "="*70)
    print("Starting Comprehensive Exploratory Data Analysis (EDA) on train_df")
    print("="*70)

    # Basic Structure and Statistics
    print("\n--- 3a. DataFrame Structure and Head ---")
    print("First 5 rows of train_df:")
    print(train_df.head())
    print("\nShape of train_df:", train_df.shape)
    print("Shape of test_df:", test_df.shape)

    print("\n--- 3b. train_df.info() (Non-null values and Data Types) ---")
    train_df.info()

    print("\n--- 3c. train_df.describe() (Descriptive Statistics for Numeric Columns) ---")
    # Display statistics for the target variable (SalePrice)
    print(train_df['SalePrice'].describe())

    # 1) Histogram of SalePrice (Raw and Log-Transformed)
    print("\n--- 3d. SalePrice Distribution (Visualizations) ---")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Distribution of House SalePrice', fontsize=18)

    # Raw SalePrice Histogram
    sns.histplot(train_df['SalePrice'], bins=50, kde=True, ax=ax1, color='teal')
    ax1.set_title('Raw SalePrice Distribution (Right-Skewed)')
    ax1.set_xlabel('SalePrice ($)', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.ticklabel_format(style='plain', axis='x')

    # Log-Transformed SalePrice Histogram (using log(1+x) to handle potential zeros, though SalePrice > 0)
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])
    sns.histplot(train_df['SalePrice_log'], bins=50, kde=True, ax=ax2, color='darkorange')
    ax2.set_title('Log-Transformed SalePrice Distribution (Closer to Normal)')
    ax2.set_xlabel('Log(1 + SalePrice)', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    # Remove the temporary log column
    train_df = train_df.drop(columns=['SalePrice_log'])


    # 2) Heatmap of Correlations among Top 10 Numeric Features related to SalePrice
    print("\n--- 3e. Correlation Heatmap (Top 10 Features vs SalePrice) ---")
    # Calculate correlations with SalePrice for all numeric columns
    corrmat = train_df.corr(numeric_only=True)
    
    # Sort features by absolute correlation value with 'SalePrice'
    # We take the top 10 most correlated features (excluding 'SalePrice' itself) and add 'SalePrice' back.
    top_features_corr = corrmat['SalePrice'].abs().sort_values(ascending=False)
    # Select the top 10 features (indices 1 to 10 are the top 10 non-SalePrice features)
    top_10_features = top_features_corr[1:11].index.tolist() + ['SalePrice']
    
    # Calculate correlation matrix for the top features
    corr_matrix_top = train_df[top_10_features].corr(numeric_only=True)

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix_top, annot=True, fmt=".2f", cmap='coolwarm', 
                linewidths=0.5, linecolor='black', cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Correlation Heatmap: Top 10 Features vs. SalePrice', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


    # 3) Boxplots for categorical variables (e.g., Neighborhood vs SalePrice)
    print("\n--- 3f. Boxplot: Neighborhood vs SalePrice ---")
    plt.figure(figsize=(18, 7))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df, palette='Spectral')
    plt.title('SalePrice Distribution Across Neighborhoods', fontsize=16)
    plt.xlabel('Neighborhood', fontsize=12)
    plt.ylabel('SalePrice ($)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()

    # 4. Data Splitting
    print("\n" + "="*70)
    print("4. Splitting Data into Training and Test Sets (80/20)")
    print("="*70)

    # Separate features (X) and target (y)
    X = train_df.drop('SalePrice', axis=1)
    y = train_df['SalePrice']

    # Perform 80/20 split (test_size=0.2 means 20% for testing)
    # We use random_state=42 for reproducibility
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Original train_df shape: {train_df.shape}")
    print(f"Features (X) shape: {X.shape}, Target (y) shape: {y.shape}")
    print("\n--- Split Shapes ---")
    print(f"X_train shape (80% features): {X_train.shape}")
    print(f"X_test shape (20% features): {X_test.shape}")
    print(f"y_train shape (80% target): {y_train.shape}")
    print(f"y_test shape (20% target): {y_test.shape}")

    # 5. Simple Linear Regression Model Training (Numeric Features Only)
    print("\n" + "="*70)
    print("5. Training a Linear Regression Model (Pre-processing Numeric Features)")
    print("="*70)
    
    # Select only numeric features for a simple initial model
    numeric_features = X_train.select_dtypes(include=np.number).columns
    
    X_train_numeric = X_train[numeric_features]
    X_test_numeric = X_test[numeric_features]

    print(f"Selected {len(numeric_features)} numeric features for modeling.")
    
    # Handle Missing Values: Impute NaNs with the median of each column
    imputer = SimpleImputer(strategy='median')
    
    # Fit imputer on X_train_numeric and transform both train and test sets
    X_train_imputed = imputer.fit_transform(X_train_numeric)
    X_test_imputed = imputer.transform(X_test_numeric)

    # Initialize and train the Linear Regression Model
    model = LinearRegression()
    print("Training Linear Regression model...")
    model.fit(X_train_imputed, y_train)
    print("Training complete.")

    # Evaluate the model on the training set (R-squared score)
    train_score = model.score(X_train_imputed, y_train)
    print(f"\nLinear Regression Training R^2 Score (Numeric Features Only): {train_score:.4f}")

    # 6. Model Evaluation on Test Set
    print("\n" + "="*70)
    print("6. Model Evaluation on Test Set")
    print("="*70)

    # Make predictions on the test set
    y_pred = model.predict(X_test_imputed)

    # 1. R^2 Score
    r2 = r2_score(y_test, y_pred)
    
    # 2. Mean Squared Error (MSE)
    mse = mean_squared_error(y_test, y_pred)

    # 3. Root Mean Squared Error (RMSE)
    rmse = np.sqrt(mse)

    print(f"R^2 Score (Test Set): {r2:.4f}")
    print(f"Mean Squared Error (MSE) (Test Set): {mse:.2f}")
    print(f"Root Mean Squared Error (RMSE) (Test Set): {rmse:.2f}")
    
else:
    print("\nSkipping EDA, Splitting, and Modeling because train_df is empty due to a loading error.")


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'Inter'

# Define the file paths as commonly found in the Kaggle environment
base_path = "/kaggle/input/house-prices-advanced-regression-techniques/"
train_file_path = os.path.join(base_path, "train.csv")
test_file_path = os.path.join(base_path, "test.csv")

# 1. Load the training data into train_df
try:
    print(f"Attempting to load training data from: {train_file_path}")
    train_df = pd.read_csv(train_file_path)
    print("Training data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Training file not found at {train_file_path}. Please ensure the path is correct.")
    # Create an empty DataFrame to prevent errors later if the file is missing
    train_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading training data: {e}")
    train_df = pd.DataFrame()

# 2. Load the test data into test_df
try:
    print(f"Attempting to load test data from: {test_file_path}")
    test_df = pd.read_csv(test_file_path)
    print("Test data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Test file not found at {test_file_path}. Please ensure the path is correct.")
    test_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading test data: {e}")
    test_df = pd.DataFrame()


# 3. Perform Exploratory Data Analysis (EDA) on train_df
if not train_df.empty:
    print("\n" + "="*70)
    print("Starting Comprehensive Exploratory Data Analysis (EDA) on train_df")
    print("="*70)

    # Basic Structure and Statistics
    print("\n--- 3a. DataFrame Structure and Head ---")
    print("First 5 rows of train_df:")
    print(train_df.head())
    print("\nShape of train_df:", train_df.shape)
    print("Shape of test_df:", test_df.shape)

    print("\n--- 3b. train_df.info() (Non-null values and Data Types) ---")
    train_df.info()

    print("\n--- 3c. train_df.describe() (Descriptive Statistics for Numeric Columns) ---")
    # Display statistics for the target variable (SalePrice)
    print(train_df['SalePrice'].describe())

    # 1) Histogram of SalePrice (Raw and Log-Transformed)
    print("\n--- 3d. SalePrice Distribution (Visualizations) ---")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Distribution of House SalePrice', fontsize=18)

    # Raw SalePrice Histogram
    sns.histplot(train_df['SalePrice'], bins=50, kde=True, ax=ax1, color='teal')
    ax1.set_title('Raw SalePrice Distribution (Right-Skewed)')
    ax1.set_xlabel('SalePrice ($)', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.ticklabel_format(style='plain', axis='x')

    # Log-Transformed SalePrice Histogram (using log(1+x) to handle potential zeros, though SalePrice > 0)
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])
    sns.histplot(train_df['SalePrice_log'], bins=50, kde=True, ax=ax2, color='darkorange')
    ax2.set_title('Log-Transformed SalePrice Distribution (Closer to Normal)')
    ax2.set_xlabel('Log(1 + SalePrice)', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    # Remove the temporary log column
    train_df = train_df.drop(columns=['SalePrice_log'])


    # 2) Heatmap of Correlations among Top 10 Numeric Features related to SalePrice
    print("\n--- 3e. Correlation Heatmap (Top 10 Features vs SalePrice) ---")
    # Calculate correlations with SalePrice for all numeric columns
    corrmat = train_df.corr(numeric_only=True)
    
    # Sort features by absolute correlation value with 'SalePrice'
    # We take the top 10 most correlated features (excluding 'SalePrice' itself) and add 'SalePrice' back.
    top_features_corr = corrmat['SalePrice'].abs().sort_values(ascending=False)
    # Select the top 10 features (indices 1 to 10 are the top 10 non-SalePrice features)
    top_10_features = top_features_corr[1:11].index.tolist() + ['SalePrice']
    
    # Calculate correlation matrix for the top features
    corr_matrix_top = train_df[top_10_features].corr(numeric_only=True)

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix_top, annot=True, fmt=".2f", cmap='coolwarm', 
                linewidths=0.5, linecolor='black', cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Correlation Heatmap: Top 10 Features vs. SalePrice', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


    # 3) Boxplots for categorical variables (e.g., Neighborhood vs SalePrice)
    print("\n--- 3f. Boxplot: Neighborhood vs SalePrice ---")
    plt.figure(figsize=(18, 7))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df, palette='Spectral')
    plt.title('SalePrice Distribution Across Neighborhoods', fontsize=16)
    plt.xlabel('Neighborhood', fontsize=12)
    plt.ylabel('SalePrice ($)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()

    # 4. Data Splitting
    print("\n" + "="*70)
    print("4. Splitting Data into Training and Test Sets (80/20)")
    print("="*70)

    # Separate features (X) and target (y)
    X = train_df.drop('SalePrice', axis=1)
    y = train_df['SalePrice']

    # Log-transform the target variable (y) now
    y = np.log1p(y)

    # Perform 80/20 split (test_size=0.2 means 20% for testing)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Original train_df shape: {train_df.shape}")
    print(f"Features (X) shape: {X.shape}, Target (y) shape: {y.shape}")
    print("\n--- Split Shapes ---")
    print(f"X_train shape (80% features): {X_train.shape}")
    print(f"X_test shape (20% features): {X_test.shape}")
    print(f"y_train shape (80% log-transformed target): {y_train.shape}")
    print(f"y_test shape (20% log-transformed target): {y_test.shape}")


    # 5. Feature Engineering and Preprocessing
    print("\n" + "="*70)
    print("5. Feature Engineering and Preprocessing (New Features and Encoding)")
    print("="*70)
    
    # 5a. Feature Combination: Total Square Footage
    X_train['TotalSF'] = X_train['GrLivArea'] + X_train['TotalBsmtSF'].fillna(0)
    X_test['TotalSF'] = X_test['GrLivArea'] + X_test['TotalBsmtSF'].fillna(0)
    print("Created new feature: 'TotalSF'.")

    # Separate feature types
    numeric_features = X_train.select_dtypes(include=np.number).columns
    categorical_features = X_train.select_dtypes(include='object').columns

    # 5b. Impute Missing Values (using median for numeric, mode for categorical)
    
    # Numeric Imputation
    num_imputer = SimpleImputer(strategy='median')
    X_train[numeric_features] = num_imputer.fit_transform(X_train[numeric_features])
    X_test[numeric_features] = num_imputer.transform(X_test[numeric_features])

    # Categorical Imputation (filling with 'Missing' to create a new category)
    X_train[categorical_features] = X_train[categorical_features].fillna('Missing')
    X_test[categorical_features] = X_test[categorical_features].fillna('Missing')
    print("Imputed missing values: median for numeric, 'Missing' for categorical.")

    # 5c. One-Hot Encoding for Categorical Features
    X_train_processed = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)
    X_test_processed = pd.get_dummies(X_test, columns=categorical_features, drop_first=True)

    # 5d. Align Columns (Crucial for consistent features between train/test after get_dummies)
    # Get common columns
    common_cols = list(set(X_train_processed.columns) & set(X_test_processed.columns))
    
    # Reindex to ensure same order and columns
    X_train_aligned = X_train_processed[common_cols]
    X_test_aligned = X_test_processed[common_cols]
    
    # Drop 'Id' column before modeling
    X_train_final = X_train_aligned.drop(columns=['Id'])
    X_test_final = X_test_aligned.drop(columns=['Id'])

    print(f"Final training feature set shape: {X_train_final.shape}")
    print(f"Final testing feature set shape: {X_test_final.shape}")


    # 6. Linear Regression Model Retraining
    print("\n" + "="*70)
    print("6. Retraining Linear Regression Model (with New Features)")
    print("="*70)
    
    # Initialize and train the Linear Regression Model
    model_improved = LinearRegression()
    print("Training Linear Regression model on new, processed features...")
    model_improved.fit(X_train_final, y_train)
    print("Training complete.")

    # Evaluate the model on the training set (R-squared score)
    train_score_improved = model_improved.score(X_train_final, y_train)
    print(f"\nImproved Linear Regression Training R^2 Score: {train_score_improved:.4f}")

    # 7. Model Re-evaluation on Test Set
    print("\n" + "="*70)
    print("7. Model Re-evaluation on Test Set (with Inverse Target Transform)")
    print("="*70)

    # Make predictions on the test set (predictions are in log scale)
    y_pred_log = model_improved.predict(X_test_final)

    # CRITICAL FIX: Clamp predictions to prevent numerical overflow (infinity) during inverse log transform.
    # We clip the log predictions to the maximum log value observed in the training data to prevent
    # the model from extrapolating wildly, which causes the ValueError.
    MAX_LOG_PREDICTION = y_train.max()
    y_pred_log = np.clip(y_pred_log, a_min=None, a_max=MAX_LOG_PREDICTION)
    print(f"Log predictions clamped: Max observed log(SalePrice) in training set was {MAX_LOG_PREDICTION:.4f}")

    # Inverse-transform the predictions and the true values to the original dollar scale for MSE/RMSE
    y_pred = np.expm1(y_pred_log)
    y_test_original = np.expm1(y_test)
    
    # 1. R^2 Score (Calculated on log-transformed values as is conventional)
    r2_improved = r2_score(y_test, y_pred_log)
    
    # 2. Mean Squared Error (MSE) (Calculated on original scale for interpretability)
    mse_improved = mean_squared_error(y_test_original, y_pred)

    # 3. Root Mean Squared Error (RMSE) (Calculated on original scale for interpretability)
    rmse_improved = np.sqrt(mse_improved)

    print(f"R^2 Score (Test Set, Log Scale): {r2_improved:.4f}")
    print(f"Mean Squared Error (MSE) (Test Set, Original Scale): {mse_improved:.2f}")
    print(f"Root Mean Squared Error (RMSE) (Test Set, Original Scale): {rmse_improved:.2f}")
    
else:
    print("\nSkipping EDA, Splitting, and Modeling because train_df is empty due to a loading error.")


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge # Added Ridge
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'Inter'

# Define the file paths as commonly found in the Kaggle environment
base_path = "/kaggle/input/house-prices-advanced-regression-techniques/"
train_file_path = os.path.join(base_path, "train.csv")
test_file_path = os.path.join(base_path, "test.csv")

# 1. Load the training data into train_df
try:
    print(f"Attempting to load training data from: {train_file_path}")
    train_df = pd.read_csv(train_file_path)
    print("Training data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Training file not found at {train_file_path}. Please ensure the path is correct.")
    # Create an empty DataFrame to prevent errors later if the file is missing
    train_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading training data: {e}")
    train_df = pd.DataFrame()

# 2. Load the test data into test_df
try:
    print(f"Attempting to load test data from: {test_file_path}")
    test_df = pd.read_csv(test_file_path)
    print("Test data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Test file not found at {test_file_path}. Please ensure the path is correct.")
    test_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading test data: {e}")
    test_df = pd.DataFrame()


# 3. Perform Exploratory Data Analysis (EDA) on train_df
if not train_df.empty:
    print("\n" + "="*70)
    print("Starting Comprehensive Exploratory Data Analysis (EDA) on train_df")
    print("="*70)

    # Basic Structure and Statistics
    print("\n--- 3a. DataFrame Structure and Head ---")
    print("First 5 rows of train_df:")
    print(train_df.head())
    print("\nShape of train_df:", train_df.shape)
    print("Shape of test_df:", test_df.shape)

    print("\n--- 3b. train_df.info() (Non-null values and Data Types) ---")
    train_df.info()

    print("\n--- 3c. train_df.describe() (Descriptive Statistics for Numeric Columns) ---")
    # Display statistics for the target variable (SalePrice)
    print(train_df['SalePrice'].describe())

    # 1) Histogram of SalePrice (Raw and Log-Transformed)
    print("\n--- 3d. SalePrice Distribution (Visualizations) ---")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Distribution of House SalePrice', fontsize=18)

    # Raw SalePrice Histogram
    sns.histplot(train_df['SalePrice'], bins=50, kde=True, ax=ax1, color='teal')
    ax1.set_title('Raw SalePrice Distribution (Right-Skewed)')
    ax1.set_xlabel('SalePrice ($)', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.ticklabel_format(style='plain', axis='x')

    # Log-Transformed SalePrice Histogram (using log(1+x) to handle potential zeros, though SalePrice > 0)
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])
    sns.histplot(train_df['SalePrice_log'], bins=50, kde=True, ax=ax2, color='darkorange')
    ax2.set_title('Log-Transformed SalePrice Distribution (Closer to Normal)')
    ax2.set_xlabel('Log(1 + SalePrice)', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    # Remove the temporary log column
    train_df = train_df.drop(columns=['SalePrice_log'])


    # 2) Heatmap of Correlations among Top 10 Numeric Features related to SalePrice
    print("\n--- 3e. Correlation Heatmap (Top 10 Features vs SalePrice) ---")
    # Calculate correlations with SalePrice for all numeric columns
    corrmat = train_df.corr(numeric_only=True)
    
    # Sort features by absolute correlation value with 'SalePrice'
    # We take the top 10 most correlated features (excluding 'SalePrice' itself) and add 'SalePrice' back.
    top_features_corr = corrmat['SalePrice'].abs().sort_values(ascending=False)
    # Select the top 10 features (indices 1 to 10 are the top 10 non-SalePrice features)
    top_10_features = top_features_corr[1:11].index.tolist() + ['SalePrice']
    
    # Calculate correlation matrix for the top features
    corr_matrix_top = train_df[top_10_features].corr(numeric_only=True)

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix_top, annot=True, fmt=".2f", cmap='coolwarm', 
                linewidths=0.5, linecolor='black', cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Correlation Heatmap: Top 10 Features vs. SalePrice', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


    # 3) Boxplots for categorical variables (e.g., Neighborhood vs SalePrice)
    print("\n--- 3f. Boxplot: Neighborhood vs SalePrice ---")
    plt.figure(figsize=(18, 7))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df, palette='Spectral')
    plt.title('SalePrice Distribution Across Neighborhoods', fontsize=16)
    plt.xlabel('Neighborhood', fontsize=12)
    plt.ylabel('SalePrice ($)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()

    # 4. Data Splitting
    print("\n" + "="*70)
    print("4. Splitting Data into Training and Test Sets (80/20)")
    print("="*70)

    # Separate features (X) and target (y)
    X = train_df.drop('SalePrice', axis=1)
    y = train_df['SalePrice']

    # Log-transform the target variable (y) now
    y = np.log1p(y)

    # Perform 80/20 split (test_size=0.2 means 20% for testing)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Original train_df shape: {train_df.shape}")
    print(f"Features (X) shape: {X.shape}, Target (y) shape: {y.shape}")
    print("\n--- Split Shapes ---")
    print(f"X_train shape (80% features): {X_train.shape}")
    print(f"X_test shape (20% features): {X_test.shape}")
    print(f"y_train shape (80% log-transformed target): {y_train.shape}")
    print(f"y_test shape (20% log-transformed target): {y_test.shape}")


    # 5. Feature Engineering and Preprocessing
    print("\n" + "="*70)
    print("5. Feature Engineering and Preprocessing (New Features and Encoding)")
    print("="*70)
    
    # 5a. Feature Combination: Total Square Footage
    X_train['TotalSF'] = X_train['GrLivArea'] + X_train['TotalBsmtSF'].fillna(0)
    X_test['TotalSF'] = X_test['GrLivArea'] + X_test['TotalBsmtSF'].fillna(0)
    print("Created new feature: 'TotalSF'.")

    # Separate feature types
    numeric_features = X_train.select_dtypes(include=np.number).columns
    categorical_features = X_train.select_dtypes(include='object').columns

    # 5b. Impute Missing Values (using median for numeric, mode for categorical)
    
    # Numeric Imputation
    num_imputer = SimpleImputer(strategy='median')
    X_train[numeric_features] = num_imputer.fit_transform(X_train[numeric_features])
    X_test[numeric_features] = num_imputer.transform(X_test[numeric_features])

    # Categorical Imputation (filling with 'Missing' to create a new category)
    X_train[categorical_features] = X_train[categorical_features].fillna('Missing')
    X_test[categorical_features] = X_test[categorical_features].fillna('Missing')
    print("Imputed missing values: median for numeric, 'Missing' for categorical.")

    # 5c. One-Hot Encoding for Categorical Features
    X_train_processed = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)
    X_test_processed = pd.get_dummies(X_test, columns=categorical_features, drop_first=True)

    # 5d. Align Columns (Crucial for consistent features between train/test after get_dummies)
    # Get common columns
    common_cols = list(set(X_train_processed.columns) & set(X_test_processed.columns))
    
    # Reindex to ensure same order and columns
    X_train_aligned = X_train_processed[common_cols]
    X_test_aligned = X_test_processed[common_cols]
    
    # Drop 'Id' column before modeling
    X_train_final = X_train_aligned.drop(columns=['Id'])
    X_test_final = X_test_aligned.drop(columns=['Id'])

    print(f"Final training feature set shape: {X_train_final.shape}")
    print(f"Final testing feature set shape: {X_test_final.shape}")


    # 6. Ridge Regression Model Retraining (New Solution)
    print("\n" + "="*70)
    print("6. Retraining with RIDGE REGRESSION (with New Features)")
    print("="*70)
    
    # Initialize and train the Ridge Regression Model (using alpha=10 for initial regularization)
    # Ridge helps stabilize the model against multicollinearity introduced by many features/OHE.
    model_improved = Ridge(alpha=10.0, random_state=42) 
    print("Training Ridge Regression model on new, processed features...")
    model_improved.fit(X_train_final, y_train)
    print("Training complete.")

    # Evaluate the model on the training set (R-squared score)
    train_score_improved = model_improved.score(X_train_final, y_train)
    print(f"\nImproved Ridge Regression Training R^2 Score: {train_score_improved:.4f}")

    # 7. Model Re-evaluation on Test Set
    print("\n" + "="*70)
    print("7. Model Re-evaluation on Test Set (with Inverse Target Transform)")
    print("="*70)

    # Make predictions on the test set (predictions are in log scale)
    y_pred_log = model_improved.predict(X_test_final)

    # CRITICAL FIX: Clamp predictions to prevent numerical overflow (infinity) during inverse log transform.
    # We clip the log predictions to the maximum log value observed in the training data to prevent
    # the model from extrapolating wildly, which causes the ValueError.
    MAX_LOG_PREDICTION = y_train.max()
    y_pred_log = np.clip(y_pred_log, a_min=None, a_max=MAX_LOG_PREDICTION)
    print(f"Log predictions clamped: Max observed log(SalePrice) in training set was {MAX_LOG_PREDICTION:.4f}")

    # Inverse-transform the predictions and the true values to the original dollar scale for MSE/RMSE
    y_pred = np.expm1(y_pred_log)
    y_test_original = np.expm1(y_test)
    
    # 1. R^2 Score (Calculated on log-transformed values as is conventional)
    r2_improved = r2_score(y_test, y_pred_log)
    
    # 2. Mean Squared Error (MSE) (Calculated on original scale for interpretability)
    mse_improved = mean_squared_error(y_test_original, y_pred)

    # 3. Root Mean Squared Error (RMSE) (Calculated on original scale for interpretability)
    rmse_improved = np.sqrt(mse_improved)

    print(f"R^2 Score (Test Set, Log Scale): {r2_improved:.4f}")
    print(f"Mean Squared Error (MSE) (Test Set, Original Scale): {mse_improved:.2f}")
    print(f"Root Mean Squared Error (RMSE) (Test Set, Original Scale): {rmse_improved:.2f}")
    
else:
    print("\nSkipping EDA, Splitting, and Modeling because train_df is empty due to a loading error.")


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge # Removed LinearRegression as it's not used
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor # Added ensemble models
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
# Note: XGBoost is often available in data science environments like Kaggle, but requires separate import
try:
    from xgboost import XGBRegressor 
    XGB_AVAILABLE = True
except ImportError:
    print("Warning: XGBoost not installed. Skipping XGBoost model.")
    XGB_AVAILABLE = False


# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'Inter'

# Define the file paths as commonly found in the Kaggle environment
base_path = "/kaggle/input/house-prices-advanced-regression-techniques/"
train_file_path = os.path.join(base_path, "train.csv")
test_file_path = os.path.join(base_path, "test.csv")

# 1. Load the training data into train_df
try:
    print(f"Attempting to load training data from: {train_file_path}")
    train_df = pd.read_csv(train_file_path)
    print("Training data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Training file not found at {train_file_path}. Please ensure the path is correct.")
    # Create an empty DataFrame to prevent errors later if the file is missing
    train_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading training data: {e}")
    train_df = pd.DataFrame()

# 2. Load the test data into test_df
try:
    print(f"Attempting to load test data from: {test_file_path}")
    test_df = pd.read_csv(test_file_path)
    print("Test data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Test file not found at {test_file_path}. Please ensure the path is correct.")
    test_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading test data: {e}")
    test_df = pd.DataFrame()


# 3. Perform Exploratory Data Analysis (EDA) on train_df
if not train_df.empty:
    print("\n" + "="*70)
    print("Starting Comprehensive Exploratory Data Analysis (EDA) on train_df")
    print("="*70)

    # Basic Structure and Statistics
    print("\n--- 3a. DataFrame Structure and Head ---")
    print("First 5 rows of train_df:")
    print(train_df.head())
    print("\nShape of train_df:", train_df.shape)
    print("Shape of test_df:", test_df.shape)

    print("\n--- 3b. train_df.info() (Non-null values and Data Types) ---")
    train_df.info()

    print("\n--- 3c. train_df.describe() (Descriptive Statistics for Numeric Columns) ---")
    # Display statistics for the target variable (SalePrice)
    print(train_df['SalePrice'].describe())

    # 1) Histogram of SalePrice (Raw and Log-Transformed)
    print("\n--- 3d. SalePrice Distribution (Visualizations) ---")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Distribution of House SalePrice', fontsize=18)

    # Raw SalePrice Histogram
    sns.histplot(train_df['SalePrice'], bins=50, kde=True, ax=ax1, color='teal')
    ax1.set_title('Raw SalePrice Distribution (Right-Skewed)')
    ax1.set_xlabel('SalePrice ($)', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.ticklabel_format(style='plain', axis='x')

    # Log-Transformed SalePrice Histogram (using log(1+x) to handle potential zeros, though SalePrice > 0)
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])
    sns.histplot(train_df['SalePrice_log'], bins=50, kde=True, ax=ax2, color='darkorange')
    ax2.set_title('Log-Transformed SalePrice Distribution (Closer to Normal)')
    ax2.set_xlabel('Log(1 + SalePrice)', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    # Remove the temporary log column
    train_df = train_df.drop(columns=['SalePrice_log'])


    # 2) Heatmap of Correlations among Top 10 Numeric Features related to SalePrice
    print("\n--- 3e. Correlation Heatmap (Top 10 Features vs SalePrice) ---")
    # Calculate correlations with SalePrice for all numeric columns
    corrmat = train_df.corr(numeric_only=True)
    
    # Sort features by absolute correlation value with 'SalePrice'
    # We take the top 10 most correlated features (excluding 'SalePrice' itself) and add 'SalePrice' back.
    top_features_corr = corrmat['SalePrice'].abs().sort_values(ascending=False)
    # Select the top 10 features (indices 1 to 10 are the top 10 non-SalePrice features)
    top_10_features = top_features_corr[1:11].index.tolist() + ['SalePrice']
    
    # Calculate correlation matrix for the top features
    corr_matrix_top = train_df[top_10_features].corr(numeric_only=True)

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix_top, annot=True, fmt=".2f", cmap='coolwarm', 
                linewidths=0.5, linecolor='black', cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Correlation Heatmap: Top 10 Features vs. SalePrice', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


    # 3) Boxplots for categorical variables (e.g., Neighborhood vs SalePrice)
    print("\n--- 3f. Boxplot: Neighborhood vs SalePrice ---")
    plt.figure(figsize=(18, 7))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df, palette='Spectral')
    plt.title('SalePrice Distribution Across Neighborhoods', fontsize=16)
    plt.xlabel('Neighborhood', fontsize=12)
    plt.ylabel('SalePrice ($)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()

    # 4. Data Splitting
    print("\n" + "="*70)
    print("4. Splitting Data into Training and Test Sets (80/20)")
    print("="*70)

    # Separate features (X) and target (y)
    X = train_df.drop('SalePrice', axis=1)
    y = train_df['SalePrice']

    # Log-transform the target variable (y) now
    y = np.log1p(y)

    # Perform 80/20 split (test_size=0.2 means 20% for testing)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Original train_df shape: {train_df.shape}")
    print(f"Features (X) shape: {X.shape}, Target (y) shape: {y.shape}")
    print("\n--- Split Shapes ---")
    print(f"X_train shape (80% features): {X_train.shape}")
    print(f"X_test shape (20% features): {X_test.shape}")
    print(f"y_train shape (80% log-transformed target): {y_train.shape}")
    print(f"y_test shape (20% log-transformed target): {y_test.shape}")


    # 5. Feature Engineering and Preprocessing
    print("\n" + "="*70)
    print("5. Feature Engineering and Preprocessing (New Features and Encoding)")
    print("="*70)
    
    # 5a. Feature Combination: Total Square Footage
    X_train['TotalSF'] = X_train['GrLivArea'] + X_train['TotalBsmtSF'].fillna(0)
    X_test['TotalSF'] = X_test['GrLivArea'] + X_test['TotalBsmtSF'].fillna(0)
    print("Created new feature: 'TotalSF'.")

    # Separate feature types
    numeric_features = X_train.select_dtypes(include=np.number).columns
    categorical_features = X_train.select_dtypes(include='object').columns

    # 5b. Impute Missing Values (using median for numeric, mode for categorical)
    
    # Numeric Imputation
    num_imputer = SimpleImputer(strategy='median')
    X_train[numeric_features] = num_imputer.fit_transform(X_train[numeric_features])
    X_test[numeric_features] = num_imputer.transform(X_test[numeric_features])

    # Categorical Imputation (filling with 'Missing' to create a new category)
    X_train[categorical_features] = X_train[categorical_features].fillna('Missing')
    X_test[categorical_features] = X_test[categorical_features].fillna('Missing')
    print("Imputed missing values: median for numeric, 'Missing' for categorical.")

    # 5c. One-Hot Encoding for Categorical Features
    X_train_processed = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)
    X_test_processed = pd.get_dummies(X_test, columns=categorical_features, drop_first=True)

    # 5d. Align Columns (Crucial for consistent features between train/test after get_dummies)
    # Get common columns
    common_cols = list(set(X_train_processed.columns) & set(X_test_processed.columns))
    
    # Reindex to ensure same order and columns
    X_train_aligned = X_train_processed[common_cols]
    X_test_aligned = X_test_processed[common_cols]
    
    # Drop 'Id' column before modeling
    X_train_final = X_train_aligned.drop(columns=['Id'])
    X_test_final = X_test_aligned.drop(columns=['Id'])

    print(f"Final training feature set shape: {X_train_final.shape}")
    print(f"Final testing feature set shape: {X_test_final.shape}")


    # 6. Initialize Models for Comparison
    print("\n" + "="*70)
    print("6. Initializing Regression Models for Comparison")
    print("="*70)
    
    # Define models with fixed random state for reproducibility
    models = [
        ('Ridge Regression (Alpha=10)', Ridge(alpha=10.0, random_state=42)),
        ('Random Forest', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
        ('Gradient Boosting', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)),
    ]
    
    if XGB_AVAILABLE:
        # XGBoost requires the objective to be explicitly set for regression
        models.append(('XGBoost', XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, objective='reg:squarederror')))
    
    results = {}
    MAX_LOG_PREDICTION = y_train.max()
    print(f"Using max log(SalePrice) in training set for clamping: {MAX_LOG_PREDICTION:.4f}")


    # 7. Train and Evaluate All Models
    print("\n" + "="*70)
    print("7. Training and Evaluation Results")
    print("="*70)
    
    # Inverse-transform the true values once for all evaluations
    y_test_original = np.expm1(y_test)

    for name, model in models:
        print(f"\n--- Training {name} ---")
        
        # Train the model
        model.fit(X_train_final, y_train)

        # Evaluate the model on the training set
        train_score = model.score(X_train_final, y_train)

        # Make predictions on the test set (predictions are in log scale)
        y_pred_log = model.predict(X_test_final)

        # Clamp predictions to prevent numerical overflow (infinity) during inverse log transform
        y_pred_log = np.clip(y_pred_log, a_min=None, a_max=MAX_LOG_PREDICTION)

        # Inverse-transform the predictions to the original dollar scale
        y_pred = np.expm1(y_pred_log)
        
        # 1. R^2 Score (Calculated on log-transformed values)
        r2_score_val = r2_score(y_test, y_pred_log)
        
        # 2. Root Mean Squared Error (RMSE) (Calculated on original scale)
        rmse_val = np.sqrt(mean_squared_error(y_test_original, y_pred))
        
        results[name] = rmse_val
        
        print(f"Training R^2 Score: {train_score:.4f}")
        print(f"Validation R^2 Score (Log Scale): {r2_score_val:.4f}")
        print(f"Validation RMSE (Original Scale): ${rmse_val:.2f}")

    # Final Summary
    print("\n" + "="*70)
    print("Model Comparison Summary (Validation RMSE)")
    print("="*70)
    
    # Sort results by RMSE (ascending)
    sorted_results = dict(sorted(results.items(), key=lambda item: item[1]))

    for name, rmse in sorted_results.items():
        print(f"  {name:<30}: ${rmse:.2f}")

    print("---")
    print("The goal is to minimize the RMSE.")
    
else:
    print("\nSkipping EDA, Splitting, and Modeling because train_df is empty due to a loading error.")


In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
# Note: XGBoost is often available in data science environments like Kaggle, but requires separate import
try:
    from xgboost import XGBRegressor 
    XGB_AVAILABLE = True
except ImportError:
    print("Warning: XGBoost not installed. Skipping XGBoost model.")
    XGB_AVAILABLE = False


# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'Inter'

# Define the file paths as commonly found in the Kaggle environment
base_path = "/kaggle/input/house-prices-advanced-regression-techniques/"
train_file_path = os.path.join(base_path, "train.csv")
test_file_path = os.path.join(base_path, "test.csv")

# 1. Load the training data into train_df
try:
    print(f"Attempting to load training data from: {train_file_path}")
    train_df = pd.read_csv(train_file_path)
    print("Training data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Training file not found at {train_file_path}. Please ensure the path is correct.")
    # Create an empty DataFrame to prevent errors later if the file is missing
    train_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading training data: {e}")
    train_df = pd.DataFrame()

# 2. Load the test data into test_df
try:
    print(f"Attempting to load test data from: {test_file_path}")
    test_df = pd.read_csv(test_file_path)
    print("Test data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Test file not found at {test_file_path}. Please ensure the path is correct.")
    test_df = pd.DataFrame()
except Exception as e:
    print(f"An unexpected error occurred while loading test data: {e}")
    test_df = pd.DataFrame()


# 3. Perform Exploratory Data Analysis (EDA) on train_df
if not train_df.empty:
    print("\n" + "="*70)
    print("Starting Comprehensive Exploratory Data Analysis (EDA) on train_df")
    print("="*70)

    # Basic Structure and Statistics
    print("\n--- 3a. DataFrame Structure and Head ---")
    print("First 5 rows of train_df:")
    print(train_df.head())
    print("\nShape of train_df:", train_df.shape)
    print("Shape of test_df:", test_df.shape)

    print("\n--- 3b. train_df.info() (Non-null values and Data Types) ---")
    train_df.info()

    # 3c. train_df.describe() (Descriptive Statistics for Numeric Columns) ---")
    # Display statistics for the target variable (SalePrice)
    print(train_df['SalePrice'].describe())

    # 1) Histogram of SalePrice (Raw and Log-Transformed)
    print("\n--- 3d. SalePrice Distribution (Visualizations) ---")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Distribution of House SalePrice', fontsize=18)

    # Raw SalePrice Histogram
    sns.histplot(train_df['SalePrice'], bins=50, kde=True, ax=ax1, color='teal')
    ax1.set_title('Raw SalePrice Distribution (Right-Skewed)')
    ax1.set_xlabel('SalePrice ($)', fontsize=12)
    ax1.set_ylabel('Frequency', fontsize=12)
    ax1.ticklabel_format(style='plain', axis='x')

    # Log-Transformed SalePrice Histogram (using log(1+x) to handle potential zeros, though SalePrice > 0)
    train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])
    sns.histplot(train_df['SalePrice_log'], bins=50, kde=True, ax=ax2, color='darkorange')
    ax2.set_title('Log-Transformed SalePrice Distribution (Closer to Normal)')
    ax2.set_xlabel('Log(1 + SalePrice)', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    # Remove the temporary log column
    train_df = train_df.drop(columns=['SalePrice_log'])


    # 2) Heatmap of Correlations among Top 10 Numeric Features related to SalePrice
    print("\n--- 3e. Correlation Heatmap (Top 10 Features vs SalePrice) ---")
    # Calculate correlations with SalePrice for all numeric columns
    corrmat = train_df.corr(numeric_only=True)
    
    # Sort features by absolute correlation value with 'SalePrice'
    # We take the top 10 most correlated features (excluding 'SalePrice' itself) and add 'SalePrice' back.
    top_features_corr = corrmat['SalePrice'].abs().sort_values(ascending=False)
    # Select the top 10 features (indices 1 to 10 are the top 10 non-SalePrice features)
    top_10_features = top_features_corr[1:11].index.tolist() + ['SalePrice']
    
    # Calculate correlation matrix for the top features
    corr_matrix_top = train_df[top_10_features].corr(numeric_only=True)

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix_top, annot=True, fmt=".2f", cmap='coolwarm', 
                linewidths=0.5, linecolor='black', cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Correlation Heatmap: Top 10 Features vs. SalePrice', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


    # 3) Boxplots for categorical variables (e.g., Neighborhood vs SalePrice)
    print("\n--- 3f. Boxplot: Neighborhood vs SalePrice ---")
    plt.figure(figsize=(18, 7))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df, palette='Spectral')
    plt.title('SalePrice Distribution Across Neighborhoods', fontsize=16)
    plt.xlabel('Neighborhood', fontsize=12)
    plt.ylabel('SalePrice ($)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.show()

    # 4. Data Splitting
    print("\n" + "="*70)
    print("4. Splitting Data into Training and Validation Sets (80/20)")
    print("="*70)

    # Separate features (X) and target (y)
    X = train_df.drop('SalePrice', axis=1)
    y = train_df['SalePrice']

    # Log-transform the target variable (y) now
    y = np.log1p(y)

    # Perform 80/20 split (test_size=0.2 means 20% for validation)
    # Renaming to X_val, y_val to distinguish from external test_df
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Store Id for the external test set for submission
    X_submission = test_df.drop(columns=['Id'], errors='ignore')
    submission_ids = test_df['Id'] # Keep track of the IDs for the final file

    print(f"Original train_df shape: {train_df.shape}")
    print(f"External test_df shape: {test_df.shape}")
    print("\n--- Split Shapes ---")
    print(f"X_train shape (80% features): {X_train.shape}")
    print(f"X_val shape (20% features - Validation): {X_val.shape}")
    print(f"y_train shape (80% log-transformed target): {y_train.shape}")
    print(f"y_val shape (20% log-transformed target - Validation): {y_val.shape}")


    # 5. Feature Engineering and Preprocessing
    print("\n" + "="*70)
    print("5. Feature Engineering and Preprocessing (Consistency across Train, Validation, and Submission Sets)")
    print("="*70)
    
    # 5a. Feature Combination: Total Square Footage
    # FIX: Changed 'TotalBasementSF' to the correct column name 'TotalBsmtSF'
    X_train['TotalSF'] = X_train['GrLivArea'] + X_train['TotalBsmtSF'].fillna(0)
    X_val['TotalSF'] = X_val['GrLivArea'] + X_val['TotalBsmtSF'].fillna(0)
    X_submission['TotalSF'] = X_submission['GrLivArea'] + X_submission['TotalBsmtSF'].fillna(0)
    print("Created new feature: 'TotalSF' across all datasets.")

    # Separate feature types
    numeric_features = X_train.select_dtypes(include=np.number).columns
    categorical_features = X_train.select_dtypes(include='object').columns
    
    # FIX: Remove the 'Id' column from the numeric features list before imputation
    # as it's an identifier and X_submission doesn't contain it.
    if 'Id' in numeric_features:
        numeric_features = numeric_features.drop('Id')

    # 5b. Impute Missing Values (using median for numeric, mode for categorical)
    
    # Numeric Imputation (Fit on Train, Transform all)
    num_imputer = SimpleImputer(strategy='median')
    X_train[numeric_features] = num_imputer.fit_transform(X_train[numeric_features])
    X_val[numeric_features] = num_imputer.transform(X_val[numeric_features])
    X_submission[numeric_features] = num_imputer.transform(X_submission[numeric_features]) # Transform submission set
    
    # Categorical Imputation (filling with 'Missing' to create a new category)
    X_train[categorical_features] = X_train[categorical_features].fillna('Missing')
    X_val[categorical_features] = X_val[categorical_features].fillna('Missing')
    X_submission[categorical_features] = X_submission[categorical_features].fillna('Missing') # Impute submission set
    print("Imputed missing values: median for numeric, 'Missing' for categorical.")

    # 5c. One-Hot Encoding for Categorical Features
    X_train_processed = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)
    X_val_processed = pd.get_dummies(X_val, columns=categorical_features, drop_first=True)
    X_submission_processed = pd.get_dummies(X_submission, columns=categorical_features, drop_first=True) # Process submission set

    # 5d. Align Columns (Crucial for consistent features)
    # Columns present in the training set define the feature space
    train_cols = X_train_processed.columns
    
    # Reindex validation and submission sets to match training columns
    # Fill missing columns (new categories not seen in train) with 0
    X_val_final = X_val_processed.reindex(columns=train_cols, fill_value=0)
    X_submission_final = X_submission_processed.reindex(columns=train_cols, fill_value=0)
    
    # Drop 'Id' column from all final feature sets
    X_train_final = X_train_processed.drop(columns=['Id'])
    X_val_final = X_val_final.drop(columns=['Id'])
    X_submission_final = X_submission_final.drop(columns=['Id']) # Drop from submission set

    print(f"Final training feature set shape: {X_train_final.shape}")
    print(f"Final validation feature set shape: {X_val_final.shape}")
    print(f"Final submission feature set shape: {X_submission_final.shape}")


    # 6. Initialize Models for Comparison
    print("\n" + "="*70)
    print("6. Initializing Regression Models for Comparison")
    print("="*70)
    
    # Define models with fixed random state for reproducibility
    models = [
        ('Ridge Regression (Alpha=10)', Ridge(alpha=10.0, random_state=42)),
        ('Random Forest', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
        ('Gradient Boosting', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)),
    ]
    
    if XGB_AVAILABLE:
        # XGBoost requires the objective to be explicitly set for regression
        models.append(('XGBoost', XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, objective='reg:squarederror')))
    
    results = {}
    MAX_LOG_PREDICTION = y_train.max()
    print(f"Using max log(SalePrice) in training set for clamping: {MAX_LOG_PREDICTION:.4f}")


    # 7. Train and Evaluate All Models (Validation Results)
    print("\n" + "="*70)
    print("7. Training and Validation Results")
    print("="*70)
    
    # Inverse-transform the true values once for all evaluations
    y_val_original = np.expm1(y_val)

    for name, model in models:
        print(f"\n--- Training {name} ---")
        
        # Train the model
        model.fit(X_train_final, y_train)

        # Evaluate the model on the training set
        train_score = model.score(X_train_final, y_train)

        # Make predictions on the validation set (predictions are in log scale)
        y_pred_log = model.predict(X_val_final)

        # Clamp predictions to prevent numerical overflow (infinity) during inverse log transform
        y_pred_log = np.clip(y_pred_log, a_min=None, a_max=MAX_LOG_PREDICTION)

        # Inverse-transform the predictions to the original dollar scale
        y_pred = np.expm1(y_pred_log)
        
        # 1. R^2 Score (Calculated on log-transformed values)
        r2_score_val = r2_score(y_val, y_pred_log)
        
        # 2. Root Mean Squared Error (RMSE) (Calculated on original scale)
        rmse_val = np.sqrt(mean_squared_error(y_val_original, y_pred))
        
        results[name] = rmse_val
        
        print(f"Training R^2 Score: {train_score:.4f}")
        print(f"Validation R^2 Score (Log Scale): {r2_score_val:.4f}")
        print(f"Validation RMSE (Original Scale): ${rmse_val:.2f}")

    # Final Summary
    print("\n" + "="*70)
    print("Model Comparison Summary (Validation RMSE)")
    print("="*70)
    
    # Sort results by RMSE (ascending)
    sorted_results = dict(sorted(results.items(), key=lambda item: item[1]))

    for name, rmse in sorted_results.items():
        print(f"  {name:<30}: ${rmse:.2f}")

    print("---")
    print("The goal is to minimize the RMSE.")
    
    # 8. Final Model Training (XGBoost) and Submission File Generation
    
    # Check if XGBoost is available and was the best performing model
    best_model_name = list(sorted_results.keys())[0]
    
    if XGB_AVAILABLE and best_model_name.startswith('XGBoost'):
        print("\n" + "="*70)
        print("8. Final XGBoost Model Training and Submission Generation")
        print("="*70)

        # Combine all training data (X_train and X_val) for final training
        X_full_train = pd.concat([X_train_final, X_val_final], ignore_index=True)
        y_full_train = pd.concat([y_train, y_val], ignore_index=True)

        print(f"Training final XGBoost model on full training set (N={X_full_train.shape[0]})")
        
        final_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, objective='reg:squarederror')
        final_model.fit(X_full_train, y_full_train)
        print("Final model training complete.")

        # Make predictions on the external test set (X_submission_final)
        print("Making predictions on the external test set...")
        final_log_predictions = final_model.predict(X_submission_final)

        # Apply the same clamping as before to prevent extreme outliers
        final_log_predictions = np.clip(final_log_predictions, a_min=None, a_max=MAX_LOG_PREDICTION)

        # Inverse-transform the predictions to the original dollar scale
        final_predictions = np.expm1(final_log_predictions)

        # Create the submission DataFrame
        submission_df = pd.DataFrame({'Id': submission_ids, 'SalePrice': final_predictions})
        
        # Save the submission file
        submission_file_path = "submission.csv"
        submission_df.to_csv(submission_file_path, index=False)
        print(f"\nSuccessfully generated submission file: {submission_file_path}")
        print("First 5 submission entries:")
        print(submission_df.head())
    
    elif not XGB_AVAILABLE:
         print("\nSkipping final XGBoost training as the library was not available.")

    else:
        print(f"\nSkipping final XGBoost training. The best model was determined to be {best_model_name}.")
    
else:
    print("\nSkipping EDA, Splitting, and Modeling because train_df is empty due to a loading error.")
